# Evaluacion Primer Parcial

# Optimización de rutas con TSP aplicado a Cemento Chimborazo

**Nombre** Jhoffre Moreano  
**Problema seleccionado:** Traveling Salesman Problem (TSP)  


Para cumplir con la actividad, implemento cuatro algoritmos:

- **Algoritmo Genético (GA)**
- **Colonia de Hormigas (ACO)**
- **Recocido Simulado (SA)**
- **Hill Climbing (HC)**

Además, documento la función de costo, el vecindario y los criterios de parada.

## 1. Planteamiento del problema

En mi ejemplo considero seis puntos: una bodega central y cinco clientes. El objetivo es encontrar el mejor orden de visita para reducir la distancia total recorrida.

In [1]:
# ============================================================
# 1. Importación de librerías
# ============================================================
# En esta parte importo las librerías que necesito para resolver el problema.
# random me ayuda a generar rutas aleatorias.
# math me sirve para operaciones matemáticas, por ejemplo, en Recocido Simulado.
# numpy me permite manejar la matriz de distancias de forma más ordenada.
# pandas lo uso para presentar los resultados en una tabla clara.

import random
import math
import numpy as np
import pandas as pd

# Fijo una semilla para que los resultados sean reproducibles.
# Esto significa que, si vuelvo a ejecutar el notebook, puedo obtener resultados similares.
random.seed(42)
np.random.seed(42)

## 2. Datos del problema

En este ejemplo trabajo con una bodega central y cinco clientes. Las distancias son referenciales y están expresadas en kilómetros. La matriz de distancias indica cuánto se recorre desde un punto hacia otro.

In [17]:
# ============================================================
# 2. Definición de lugares y matriz de distancias
# ============================================================
# Estos son los puntos que el trabajador de Cemento Chimborazo debe visitar.
# La Bodega Central representa el punto de partida y también el punto de retorno.

lugares = [
    "Bodega Central",
    "Cliente A",
    "Cliente B",
    "Cliente C",
    "Cliente D",
    "Cliente E"
]

# Matriz de distancias en kilómetros.
# Cada fila y cada columna representa un lugar.
# Por ejemplo, desde Bodega Central hasta Cliente A hay 10 km.
# La diagonal principal tiene cero porque no existe distancia de un lugar hacia sí mismo.

distancias = np.array([
    [0, 10, 15, 20, 18, 12],
    [10, 0, 8, 12, 15, 9],
    [15, 8, 0, 6, 10, 14],
    [20, 12, 6, 0, 7, 11],
    [18, 15, 10, 7, 0, 5],
    [12, 9, 14, 11, 5, 0]
])

# Presento la matriz como una tabla para que sea más fácil interpretarla.
tabla_distancias = pd.DataFrame(distancias, index=lugares, columns=lugares)
tabla_distancias

,Bodega Central,Cliente A,Cliente B,Cliente C,Cliente D,Cliente E
Bodega Central,0,10,15,20,18,12
Cliente A,10,0,8,12,15,9
Cliente B,15,8,0,6,10,14
Cliente C,20,12,6,0,7,11
Cliente D,18,15,10,7,0,5
Cliente E,12,9,14,11,5,0


## 3. Función de costo

La función de costo es la parte más importante del TSP. En este caso, el costo corresponde a la **distancia total recorrida**. Mientras menor sea la distancia total, mejor será la ruta encontrada.

In [18]:
# ============================================================
# 3. Función de costo
# ============================================================
# Esta función recibe una ruta y calcula la distancia total recorrida.
# La ruta es una lista con los índices de los lugares.
# Además, al final se suma la distancia para regresar al punto inicial.

def calcular_costo(ruta):
    costo = 0

    # Recorro la ruta desde el primer punto hasta el último.
    for i in range(len(ruta) - 1):
        origen = ruta[i]
        destino = ruta[i + 1]
        costo += distancias[origen][destino]

    # Aquí cierro el ciclo, es decir, regreso desde el último cliente a la bodega inicial.
    costo += distancias[ruta[-1]][ruta[0]]

    return costo


def mostrar_ruta(ruta):
    # Esta función convierte la ruta numérica en nombres de lugares.
    # La uso para que el resultado sea más entendible al momento de leerlo.
    nombres = [lugares[i] for i in ruta]
    nombres.append(lugares[ruta[0]])
    return " → ".join(nombres)

# Prueba rápida con una ruta inicial ordenada.
ruta_prueba = [0, 1, 2, 3, 4, 5]
print("Ruta de prueba:", mostrar_ruta(ruta_prueba))
print("Costo de la ruta:", calcular_costo(ruta_prueba), "km")

Ruta de prueba: Bodega Central → Cliente A → Cliente B → Cliente C → Cliente D → Cliente E → Bodega Central
Costo de la ruta: 48 km


## 4. Vecindario utilizado

El vecindario lo construyo intercambiando dos posiciones dentro de una ruta. Por ejemplo, si la ruta es Bodega → Cliente A → Cliente B → Cliente C, puedo cambiar Cliente A con Cliente C y obtener una nueva ruta candidata.

Explorando diferentes soluciones.

In [19]:
# ============================================================
# 4. Función para generar vecinos
# ============================================================
# Esta función genera una nueva ruta a partir de otra.
# Para hacerlo, selecciono dos posiciones aleatorias e intercambio sus lugares.
# Este intercambio representa el vecindario del problema.

def generar_vecino(ruta):
    nueva_ruta = ruta[:]
    i, j = random.sample(range(len(nueva_ruta)), 2)
    nueva_ruta[i], nueva_ruta[j] = nueva_ruta[j], nueva_ruta[i]
    return nueva_ruta

# Ejemplo de generación de vecino.
ruta_inicial = [0, 1, 2, 3, 4, 5]
vecino = generar_vecino(ruta_inicial)

print("Ruta original:", mostrar_ruta(ruta_inicial))
print("Ruta vecina:", mostrar_ruta(vecino))

Ruta original: Bodega Central → Cliente A → Cliente B → Cliente C → Cliente D → Cliente E → Bodega Central
Ruta vecina: Cliente C → Cliente A → Cliente B → Bodega Central → Cliente D → Cliente E → Cliente C


## 5. Algoritmo de trayectoria: Hill Climbing

Hill Climbing busca mejorar una solución paso a paso. Si encuentra una ruta vecina con menor distancia, acepta esa ruta como nueva solución. Su desventaja es que puede quedarse atrapado en una solución local.

In [20]:
# ============================================================
# 5. Hill Climbing
# ============================================================
# En este algoritmo empiezo con una ruta aleatoria.
# Luego genero vecinos intercambiando dos ciudades.
# Si el vecino mejora la distancia, lo acepto como nueva mejor ruta.
# Criterio de parada: número máximo de iteraciones.

def hill_climbing(iteraciones=1000):
    ruta = list(range(len(lugares)))
    random.shuffle(ruta)

    mejor_ruta = ruta[:]
    mejor_costo = calcular_costo(mejor_ruta)

    for _ in range(iteraciones):
        nueva_ruta = generar_vecino(mejor_ruta)
        nuevo_costo = calcular_costo(nueva_ruta)

        # Solo acepto la nueva ruta si mejora el costo actual.
        if nuevo_costo < mejor_costo:
            mejor_ruta = nueva_ruta[:]
            mejor_costo = nuevo_costo

    return mejor_ruta, mejor_costo

## 6. Algoritmo de trayectoria: Recocido Simulado

El Recocido Simulado también trabaja con rutas vecinas, pero tiene una ventaja: puede aceptar temporalmente rutas peores para evitar quedarse atrapado en una solución local. Conforme baja la temperatura, el algoritmo se vuelve más estricto.

In [21]:
# ============================================================
# 6. Recocido Simulado
# ============================================================
# Este algoritmo empieza con una ruta aleatoria.
# Acepta rutas mejores y, en algunos casos, rutas peores.
# Esto ayuda a explorar más opciones antes de decidir la mejor solución.
# Criterio de parada: número máximo de iteraciones o temperatura muy baja.

def recocido_simulado(iteraciones=1000, temperatura=100, enfriamiento=0.95):
    ruta = list(range(len(lugares)))
    random.shuffle(ruta)

    mejor_ruta = ruta[:]
    mejor_costo = calcular_costo(ruta)

    for _ in range(iteraciones):
        nueva_ruta = generar_vecino(ruta)

        costo_actual = calcular_costo(ruta)
        nuevo_costo = calcular_costo(nueva_ruta)

        # Si la nueva ruta es mejor, se acepta.
        # Si es peor, puede aceptarse dependiendo de la temperatura.
        if nuevo_costo < costo_actual or random.random() < math.exp((costo_actual - nuevo_costo) / temperatura):
            ruta = nueva_ruta[:]

        # Guardo la mejor ruta encontrada hasta el momento.
        if calcular_costo(ruta) < mejor_costo:
            mejor_ruta = ruta[:]
            mejor_costo = calcular_costo(ruta)

        # Reduzco la temperatura en cada iteración.
        temperatura *= enfriamiento

        # Evito que la temperatura llegue a cero completamente.
        if temperatura < 0.0001:
            break

    return mejor_ruta, mejor_costo

## 7. Algoritmo bioinspirado: Algoritmo Genético

El Algoritmo Genético se inspira en la evolución natural. Trabaja con una población de rutas, selecciona las mejores, cruza información entre ellas y aplica pequeñas mutaciones para encontrar mejores soluciones.

In [22]:
# ============================================================
# 7. Algoritmo Genético
# ============================================================
# En este algoritmo cada individuo representa una ruta posible.
# La población contiene varias rutas.
# En cada generación se seleccionan las mejores rutas, se cruzan y se mutan.
# Criterio de parada: número máximo de generaciones.

def algoritmo_genetico(tamano_poblacion=30, generaciones=150, tasa_mutacion=0.20):
    poblacion = []

    # Creo la población inicial con rutas aleatorias.
    for _ in range(tamano_poblacion):
        ruta = list(range(len(lugares)))
        random.shuffle(ruta)
        poblacion.append(ruta)

    for _ in range(generaciones):
        # Ordeno la población desde la ruta de menor costo hasta la de mayor costo.
        poblacion = sorted(poblacion, key=calcular_costo)

        # Conservo las mejores rutas para no perder buenas soluciones.
        nueva_poblacion = poblacion[:5]

        while len(nueva_poblacion) < tamano_poblacion:
            padre1, padre2 = random.sample(poblacion[:10], 2)

            # Cruce: tomo una parte del primer padre y completo con el segundo padre.
            punto = random.randint(1, len(lugares) - 2)
            hijo = padre1[:punto]

            for ciudad in padre2:
                if ciudad not in hijo:
                    hijo.append(ciudad)

            # Mutación: intercambio dos ciudades con cierta probabilidad.
            if random.random() < tasa_mutacion:
                hijo = generar_vecino(hijo)

            nueva_poblacion.append(hijo)

        poblacion = nueva_poblacion

    mejor_ruta = min(poblacion, key=calcular_costo)
    mejor_costo = calcular_costo(mejor_ruta)

    return mejor_ruta, mejor_costo

## 8. Algoritmo bioinspirado: Colonia de Hormigas

La Colonia de Hormigas se inspira en el comportamiento de las hormigas reales. Las hormigas dejan feromonas en los caminos que recorren. En el algoritmo, las rutas más cortas reciben más feromona y tienen más probabilidad de ser elegidas en siguientes iteraciones.

In [23]:
# ============================================================
# 8. Colonia de Hormigas
# ============================================================
# En este algoritmo varias hormigas construyen rutas.
# Cada hormiga decide el siguiente punto tomando en cuenta dos aspectos:
# 1. La feromona acumulada en el camino.
# 2. La cercanía entre puntos, es decir, menor distancia.
# Criterio de parada: número máximo de iteraciones.

def colonia_hormigas(iteraciones=100, cantidad_hormigas=15, alfa=1, beta=2, evaporacion=0.5):
    n = len(lugares)
    feromonas = np.ones((n, n))

    mejor_ruta = None
    mejor_costo = float("inf")

    for _ in range(iteraciones):
        rutas_iteracion = []

        for _ in range(cantidad_hormigas):
            inicio = random.randint(0, n - 1)
            ruta = [inicio]
            no_visitadas = set(range(n))
            no_visitadas.remove(inicio)

            while no_visitadas:
                actual = ruta[-1]
                ciudades_disponibles = list(no_visitadas)
                probabilidades = []

                for ciudad in ciudades_disponibles:
                    # La feromona representa qué tan atractivo se ha vuelto ese camino.
                    valor_feromona = feromonas[actual][ciudad] ** alfa

                    # La visibilidad favorece caminos más cortos.
                    valor_visibilidad = (1 / distancias[actual][ciudad]) ** beta

                    probabilidades.append(valor_feromona * valor_visibilidad)

                probabilidades = np.array(probabilidades)
                probabilidades = probabilidades / probabilidades.sum()

                siguiente = random.choices(ciudades_disponibles, weights=probabilidades)[0]
                ruta.append(siguiente)
                no_visitadas.remove(siguiente)

            costo = calcular_costo(ruta)
            rutas_iteracion.append((ruta, costo))

            if costo < mejor_costo:
                mejor_ruta = ruta[:]
                mejor_costo = costo

        # Evaporación de feromonas: evita que un camino domine demasiado rápido.
        feromonas *= evaporacion

        # Actualización de feromonas: las mejores rutas dejan más feromona.
        for ruta, costo in rutas_iteracion:
            for i in range(len(ruta) - 1):
                origen = ruta[i]
                destino = ruta[i + 1]
                feromonas[origen][destino] += 1 / costo
                feromonas[destino][origen] += 1 / costo

            # También actualizo el camino de retorno al inicio.
            feromonas[ruta[-1]][ruta[0]] += 1 / costo
            feromonas[ruta[0]][ruta[-1]] += 1 / costo

    return mejor_ruta, mejor_costo

## 9. Ejecución de los cuatro algoritmos

En esta sección ejecuto los cuatro algoritmos y comparo sus resultados. La mejor solución será la que tenga menor distancia total.

In [34]:
# ============================================================
# 9. Ejecución de algoritmos
# ============================================================
# Aquí ejecuto los dos algoritmos bioinspirados y los dos algoritmos de trayectoria.
# Luego guardo los resultados para compararlos en una tabla.

resultados = {}

ruta_hc, costo_hc = hill_climbing(iteraciones=1000)
resultados["Hill Climbing"] = (ruta_hc, costo_hc)

ruta_sa, costo_sa = recocido_simulado(iteraciones=1000, temperatura=100, enfriamiento=0.95)
resultados["Recocido Simulado"] = (ruta_sa, costo_sa)

ruta_ga, costo_ga = algoritmo_genetico(tamano_poblacion=30, generaciones=150, tasa_mutacion=0.20)
resultados["Algoritmo Genético"] = (ruta_ga, costo_ga)

ruta_aco, costo_aco = colonia_hormigas(iteraciones=100, cantidad_hormigas=15, alfa=1, beta=2, evaporacion=0.5)
resultados["Colonia de Hormigas"] = (ruta_aco, costo_aco)

# Presento cada resultado en pantalla.
for algoritmo, (ruta, costo) in resultados.items():
    print("Algoritmo:", algoritmo)
    print("Ruta encontrada:", mostrar_ruta(ruta))
    print("Distancia total:", costo, "km")

Algoritmo: Hill Climbing
Ruta encontrada: Cliente A → Cliente B → Cliente C → Cliente D → Cliente E → Bodega Central → Cliente A
Distancia total: 48 km
Algoritmo: Recocido Simulado
Ruta encontrada: Bodega Central → Cliente E → Cliente D → Cliente C → Cliente B → Cliente A → Bodega Central
Distancia total: 48 km
Algoritmo: Algoritmo Genético
Ruta encontrada: Cliente A → Cliente B → Cliente C → Cliente D → Cliente E → Bodega Central → Cliente A
Distancia total: 48 km
Algoritmo: Colonia de Hormigas
Ruta encontrada: Cliente E → Cliente D → Cliente C → Cliente B → Cliente A → Bodega Central → Cliente E
Distancia total: 48 km


In [35]:
# ============================================================
# 10. Tabla comparativa de resultados
# ============================================================
# En esta tabla comparo los resultados obtenidos por cada algoritmo.
# Esto me permite identificar de forma clara cuál encontró la menor distancia.

tabla_resultados = []

for algoritmo, (ruta, costo) in resultados.items():
    tabla_resultados.append({
        "Algoritmo": algoritmo,
        "Ruta encontrada": mostrar_ruta(ruta),
        "Distancia total (km)": costo
    })

df_resultados = pd.DataFrame(tabla_resultados)
df_resultados = df_resultados.sort_values(by="Distancia total (km)").reset_index(drop=True)
df_resultados

,Algoritmo,Ruta encontrada,Distancia total (km)
0,Hill Climbing,Cliente A → Cliente B → Cliente C → Cliente D ...,48
1,Recocido Simulado,Bodega Central → Cliente E → Cliente D → Clien...,48
2,Algoritmo Genético,Cliente A → Cliente B → Cliente C → Cliente D ...,48
3,Colonia de Hormigas,Cliente E → Cliente D → Cliente C → Cliente B ...,48


In [14]:
# ============================================================
# 11. Selección automática del mejor resultado
# ============================================================
# Finalmente selecciono el algoritmo que obtuvo la menor distancia total.

mejor_algoritmo = min(resultados, key=lambda nombre: resultados[nombre][1])
mejor_ruta, mejor_costo = resultados[mejor_algoritmo]

print("Mejor algoritmo encontrado:", mejor_algoritmo)
print("Mejor ruta:", mostrar_ruta(mejor_ruta))
print("Menor distancia total:", mejor_costo, "km")

Mejor algoritmo encontrado: Hill Climbing
Mejor ruta: Bodega Central → Cliente A → Cliente B → Cliente C → Cliente D → Cliente E → Bodega Central
Menor distancia total: 48 km
